[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 05](README.md)

# Modelo de offload y portabilidad

**Tema:** 05 · **Sesiones:** 22 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cuándo desplazar cómputo a un dispositivo compensa transferencia, inicialización y sincronización?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** Un acelerador puede ejecutar más cómputo por segundo y aun perder frente a la CPU cuando inicialización y transferencias dominan.

**Prerrequisitos.**

- OpenMP en CPU y jerarquía de memoria.
- Diferencia entre corrección funcional y evidencia de rendimiento.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Separar host, device y runtime de offload.
- Modelar tiempo extremo a extremo.
- Diseñar fallback CPU con corrección equivalente.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Offload portable conserva una interfaz, no rendimiento idéntico entre dispositivos.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

El costo total incluye descubrimiento, asignación, mapeo, transferencia, kernel y sincronización.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

Una ruta CPU válida permite probar corrección sin afirmar evidencia de acelerador.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- offload — delegación de cómputo a un dispositivo
- mapping — relación entre almacenamiento del host y del dispositivo
- fallback — ejecución alternativa en host que conserva corrección


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Offload Host Device

![Flujo de datos entre host y dispositivo](../../images/offload-host-device.svg)

**Cómo leerlo.** Separa preparación, H2D, kernel, D2H y validación. Esa separación evita llamar tiempo total a una medición que solo cubre el kernel.

### Metodo Rendimiento

![Ciclo de medición, resumen, perfil e hipótesis](../../images/metodo-rendimiento.svg)

**Cómo leerlo.** Una medición se repite y resume antes de perfilar. La conclusión genera un experimento nuevo cambiando una sola variable controlada.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "05"
NOTEBOOK = "05_openmp_target/01_modelo_offload.ipynb"
assert (ROOT / "curso" / "notebooks" / "05_openmp_target" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Punto de equilibrio

**Situación.** Se compara CPU con dispositivo incluyendo transferencias.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
cpu_rate = 40e9
device_rate = 600e9
link_rate = 24e9
startup = 80e-6
assert min(cpu_rate, device_rate, link_rate, startup) > 0
for bytes_ in (1e5, 1e6, 1e7, 1e8, 1e9):
    work = 4 * bytes_
    cpu = work / cpu_rate
    device = startup + 2*bytes_/link_rate + work/device_rate
    print(f"bytes={bytes_:10.0f} cpu={cpu*1e3:8.3f}ms offload={device*1e3:8.3f}ms conviene={device<cpu}")


### Explicación del resultado

El modelo debe recalibrarse con el enlace, dispositivo y kernel reales; no incluye aún solapamiento ni reutilización de datos.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Selección de ruta

**Situación.** Se hace explícita una política de fallback verificable.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
def execution_path(devices, requested=True):
    if requested and devices > 0: return "device"
    return "host-fallback"
assert execution_path(0) == "host-fallback"
assert execution_path(1) == "device"
for devices in (0, 1, 2): print(devices, execution_path(devices))


### Lectura razonada

El informe registra la ruta usada; ejecutar fallback no demuestra soporte ni rendimiento del dispositivo.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Qué tamaño o reutilización de datos desplaza el punto de equilibrio a favor del dispositivo?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Registrar número y tipo de dispositivos visibles.
2. Medir por separado transferencia y cómputo.
3. Comparar resultado con la misma referencia serial.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Cronometrar solo el kernel y llamarlo tiempo total.
- Suponer que portabilidad implica ausencia de ajustes.
- Ocultar que se ejecutó fallback CPU.


## Criterios de aceptación

- Ruta host/device registrada.
- Tiempo extremo a extremo y tiempo de kernel separados.
- Tolerancia de corrección idéntica.


## Síntesis

- La pregunta que debes poder responder es: **¿Cuándo desplazar cómputo a un dispositivo compensa transferencia, inicialización y sincronización?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Toolchain](../../../config/course-toolchain.cmake)
- [Planeación offload](../../../docs/PLANEACION_CURSO.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 05](README.md)
